# 🎙️ Kin-AI-Avatar: OpenVoice V2 + Avatar Generation in Google Colab
This notebook runs **OpenVoice V2** (instant zero-shot voice cloning) on a free **NVIDIA T4 GPU** and connects directly to your local `Kin-ai-avatar` project.

### How it works:
1. In Google Colab, select **Runtime > Change runtime type > T4 GPU**.
2. Run all cells in order.
3. Upload a 5–15 second audio clip of the person you want to clone.
4. The notebook runs a **FastAPI server** with **Ngrok**, giving you a public URL (e.g. `https://xxxx.ngrok-free.app`).
5. Put that URL into your local `Backend/.env` to stream cloned voices directly to your local application!

In [ ]:
# 1. Verify GPU allocation (Make sure you are on a T4 GPU)
!nvidia-smi
import torch
print('CUDA Available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU Device:', torch.cuda.get_device_name(0))
else:
    print('WARNING: GPU not detected! Please go to Runtime > Change runtime type > T4 GPU')


In [ ]:
# 2. Clone Repositories & Install Complete Dependencies
!rm -rf /content/OpenVoice /content/MeloTTS

# Clone both OpenVoice and MeloTTS directly
!git clone https://github.com/myshell-ai/OpenVoice.git /content/OpenVoice
!git clone https://github.com/myshell-ai/MeloTTS.git /content/MeloTTS

# Install all audio, text processing, phonemizer, and language dependencies directly
!pip install -q faster-whisper whisper-timestamped wavmark pydub librosa soundfile
!pip install -q unidecode eng_to_ipa inflect txtsplit
!pip install -q "gruut[de,es,fr]" gruut_ipa g2pkk loguru
!pip install -q unidic-lite mecab-python3 pykakasi fugashi g2p_en anyascii jamo
!pip install -q pypinyin cn2an jieba langid num2words cached_path transformers
!pip install -q fastapi uvicorn pyngrok huggingface_hub nest_asyncio

# Register both packages without running broken setup.py dependencies
!pip install -q --no-deps -e /content/OpenVoice
!pip install -q --no-deps -e /content/MeloTTS

# Pre-download required NLTK resources
import nltk
for res in ['averaged_perceptron_tagger', 'averaged_perceptron_tagger_eng', 'cmudict', 'punkt', 'punkt_tab']:
    nltk.download(res, quiet=True)

import sys
for p in ['/content/OpenVoice', '/content/MeloTTS']:
    if p not in sys.path:
        sys.path.insert(0, p)

print('All dependencies and repositories installed successfully!')


In [ ]:
# 3. Download OpenVoice V2 Checkpoints from Hugging Face
import os
from huggingface_hub import snapshot_download

ckpt_dir = '/content/OpenVoice/checkpoints_v2'
os.makedirs(ckpt_dir, exist_ok=True)

print('Downloading OpenVoice V2 weights from HuggingFace...')
snapshot_download(
    repo_id='myshell-ai/OpenVoiceV2',
    local_dir=ckpt_dir,
    allow_patterns=['converter/*', 'base_speakers/*']
)
print('Weights downloaded successfully to:', ckpt_dir)


In [ ]:
# 4. Load Models into GPU VRAM (Super Fast: ~300ms inference)
import os
import sys
for p in ['/content/OpenVoice', '/content/MeloTTS']:
    if p not in sys.path:
        sys.path.insert(0, p)

import torch
from openvoice import se_extractor
from openvoice.api import ToneColorConverter
from openvoice.mel_processing import spectrogram_torch
from melo.api import TTS

device = 'cuda:0' if torch.cuda.is_available() else 'cpu'
print(f'Loading models onto {device}...')

# Tone Color Converter
converter_path = '/content/OpenVoice/checkpoints_v2/converter'
tone_converter = ToneColorConverter(
    f'{converter_path}/config.json',
    device=device
)
tone_converter.load_ckpt(f'{converter_path}/checkpoint.pth')
tone_converter.watermark_model = None  # Disabled for maximum speed

# Base TTS model
melo_tts = TTS(language='EN', device=device)
speaker_ids = melo_tts.hps.data.spk2id

# Preload source speaker embeddings
ses_dir = '/content/OpenVoice/checkpoints_v2/base_speakers/ses'
source_ses = {
    fname.replace('.pth', ''): torch.load(os.path.join(ses_dir, fname), map_location=device)
    for fname in os.listdir(ses_dir) if fname.endswith('.pth')
}

print('Available accents:', list(source_ses.keys()))
print('OpenVoice V2 loaded successfully on GPU!')


In [ ]:
# 5. Upload Reference Audio (Accepts ANY format: WhatsApp AAC, MP3, WAV, etc.)
from google.colab import files
import os
import re
import subprocess
import torch
from openvoice import se_extractor

os.makedirs('/content/voices', exist_ok=True)
print('Upload a 5-15 second audio sample (WhatsApp AAC, MP3, WAV, M4A, etc.):')
uploaded = files.upload()

target_se_cache = {}
for fname, data in uploaded.items():
    base_name = os.path.splitext(fname)[0]
    spk_name = re.sub(r'[^a-zA-Z0-9_]', '_', base_name).strip('_') or 'elder'
    
    raw_temp = f'/content/voices/temp_{spk_name}{os.path.splitext(fname)[1]}'
    clean_wav = f'/content/voices/{spk_name}.wav'
    
    with open(raw_temp, 'wb') as f:
        f.write(data)
        
    # Convert to 22050Hz Mono WAV via ffmpeg
    print(f'Converting {fname} to clean WAV format...')
    subprocess.run([
        'ffmpeg', '-y', '-i', raw_temp,
        '-ar', '22050', '-ac', '1', clean_wav
    ], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    
    if os.path.exists(raw_temp):
        os.remove(raw_temp)
        
    print(f'Extracting tone color embedding for {spk_name}...')
    target_se, _ = se_extractor.get_se(clean_wav, tone_converter, vad=True)
    target_se_cache[spk_name] = target_se
    torch.save(target_se.cpu(), f'/content/voices/{spk_name}_se.pth')
    print(f'Done! Voice profile {spk_name} is registered and saved.')


In [ ]:
# 6. Test Voice Generation in Notebook
import soundfile as sf
import IPython.display as ipd
import os
import torch
import nltk
from openvoice.mel_processing import spectrogram_torch

# Ensure NLTK taggers are downloaded
for res in ['averaged_perceptron_tagger', 'averaged_perceptron_tagger_eng', 'cmudict', 'punkt', 'punkt_tab']:
    nltk.download(res, quiet=True)

def generate_cloned_voice(text, speaker_name, accent='en-us', speed=1.0):
    if speaker_name not in target_se_cache:
        pth = f'/content/voices/{speaker_name}_se.pth'
        if os.path.exists(pth):
            target_se_cache[speaker_name] = torch.load(pth, map_location=device)
        else:
            print(f'Error: Voice profile {speaker_name} not found!')
            return None
            
    target_se = target_se_cache[speaker_name]
    source_se = source_ses.get(accent, source_ses['en-us'])
    
    # Generate base speech in RAM
    base_audio = melo_tts.tts_to_file(text, speaker_ids['EN-US'], output_path=None, speed=speed)
    
    # Convert Tone Color in VRAM
    hps = tone_converter.hps
    audio_tensor = torch.from_numpy(base_audio).float().to(device).unsqueeze(0)
    with torch.no_grad():
        spec = spectrogram_torch(
            audio_tensor, hps.data.filter_length, hps.data.sampling_rate,
            hps.data.hop_length, hps.data.win_length, center=False
        ).to(device)
        spec_lengths = torch.LongTensor([spec.size(-1)]).to(device)
        cloned = tone_converter.model.voice_conversion(
            spec, spec_lengths, sid_src=source_se, sid_tgt=target_se, tau=0.3
        )[0][0, 0].data.cpu().float().numpy()
        
    out_path = '/content/sample_cloned.wav'
    sf.write(out_path, cloned, 22050)
    return out_path

# Pick first uploaded speaker name (or change to your filename):
test_name = list(target_se_cache.keys())[0] if target_se_cache else 'elder'
audio_file = generate_cloned_voice('Hello! I am speaking using my cloned voice on Google Colab with OpenVoice.', test_name)
if audio_file:
    display(ipd.Audio(audio_file))


In [ ]:
# 7. Start FastAPI Server & Expose via PyNgrok for your Local App
import nest_asyncio
import io
from pyngrok import ngrok
from fastapi import FastAPI, UploadFile, File, Form, HTTPException
from fastapi.responses import Response
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel
import uvicorn

nest_asyncio.apply()

app = FastAPI(title='Kin-AI-Avatar Voice Engine')
app.add_middleware(CORSMiddleware, allow_origins=['*'], allow_credentials=True, allow_methods=['*'], allow_headers=['*'])

@app.get('/health')
def health():
    return {'status': 'healthy', 'device': device, 'loaded_speakers': list(target_se_cache.keys())}

class SynthReq(BaseModel):
    text: str
    speaker_name: str = 'elder'
    accent: str = 'en-us'
    speed: float = 1.0

@app.post('/synthesize')
def api_synthesize(req: SynthReq):
    out_path = generate_cloned_voice(req.text, req.speaker_name, req.accent, req.speed)
    if not out_path:
        raise HTTPException(status_code=400, detail='Voice generation failed')
    with open(out_path, 'rb') as f:
        audio_bytes = f.read()
    return Response(content=audio_bytes, media_type='audio/wav')

# Enter your Ngrok Authtoken below (Get free token at https://dashboard.ngrok.com/get-started/your-authtoken)
NGROK_TOKEN = 'YOUR_NGROK_AUTHTOKEN_HERE'
if NGROK_TOKEN != 'YOUR_NGROK_AUTHTOKEN_HERE':
    ngrok.set_auth_token(NGROK_TOKEN)

tunnel = ngrok.connect(8000)
print('\n' + '='*65)
print(f'🚀 PUBLIC COLAB URL: {tunnel.public_url}')
print(f'Set COLAB_VOICE_URL={tunnel.public_url} in your local Kin-ai-avatar .env')
print('='*65 + '\n')

uvicorn.run(app, host='0.0.0.0', port=8000)
